In [3]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import re

# 🔧 CONFIG
PDF_PATH = r"C:\Users\IlaBarshilia\OneDrive - ldiltd.com\M1 Global - M1 Global\Projects\Job Reconciliation Analysis\PDF Input from Top Customers\CW Roberts - P&S - T2914 - thru 8.17.25 - 1124.42.pdf"
OUTPUT_TEXT_FILE = r"C:\Users\IlaBarshilia\OneDrive - ldiltd.com\M1 Global - M1 Global\Projects\Job Reconciliation Analysis\PDF Input from Top Customers\CW_Roberts_raw_text.txt"

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Users\IlaBarshilia\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"
)

POPPLER_PATH = r"C:\Users\IlaBarshilia\Downloads\Software\Poppler\poppler-25.12.0\Library\bin"

# Convert PDF → Images
pages = convert_from_path(PDF_PATH, dpi=300, poppler_path=POPPLER_PATH)

def correct_rotation(image):
    """Detect and fix rotation using Tesseract OSD"""
    try:
        osd = pytesseract.image_to_osd(image)
        rotate_angle = int(re.search(r"Rotate: (\d+)", osd).group(1))
        if rotate_angle != 0:
            image = image.rotate(-rotate_angle, expand=True)
    except Exception:
        pass
    return image

all_text = []

for idx, page in enumerate(pages, start=1):
    print(f"OCR page {idx}...")

    page = correct_rotation(page)

    text = pytesseract.image_to_string(
        page,
        config="--psm 6"  # good for tables/invoices
    )
    all_text.append(text)

# Save raw OCR output
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_text))

print("✅ Raw OCR text saved:", OUTPUT_TEXT_FILE)

OCR page 1...
OCR page 2...
OCR page 3...
✅ Raw OCR text saved: C:\Users\IlaBarshilia\OneDrive - ldiltd.com\M1 Global - M1 Global\Projects\Job Reconciliation Analysis\PDF Input from Top Customers\CW_Roberts_raw_text.txt


#### Data into Tabular form for CW Roberts

In [17]:
import re
import pandas as pd

def clean_number(x):
    return (
        x.replace("$", "")
         .replace("§", "")
         .replace(",", "")
         .strip()
    )


RAW_TEXT_FILE = r"C:\Users\IlaBarshilia\OneDrive - ldiltd.com\M1 Global - M1 Global\Projects\Job Reconciliation Analysis\PDF Input from Top Customers\CW_Roberts_raw_text.txt"

with open(RAW_TEXT_FILE, "r", encoding="utf-8") as f:
    raw_text = f.read()

amount = re.search(
    r"amount of\s*\$?\s*([\d,]+\.\d{2})",
    raw_text,
    re.IGNORECASE
)

amount = amount.group(1) if amount else None

job_name = re.search(
    r"Job Name:\s*(.+)",
    raw_text
)

job_name = job_name.group(1).strip() if job_name else None

job_address = re.search(
    r"Job Address:\s*(.*?)\s*Job Number:",
    raw_text,
    re.DOTALL
)

job_address = (
    " ".join(job_address.group(1).split())
    if job_address else None
)

job_number = re.search(
    r"Job Number:\s*([\d\.]+)",
    raw_text
)

job_number = job_number.group(1) if job_number else None

metadata = {
    "Amount": amount,
    "Job Name": job_name,
    "Job Address": job_address,
    "Job Number": job_number
}
df_metadata = pd.DataFrame([metadata])

table_block = re.search(
    r"PAY ITEM NO\.(.*?)TOTAL\s+\$",
    raw_text,
    re.DOTALL
)
table_block = table_block.group(1) if table_block else None

columns = [
    "Pay Item No",
    "Description",
    "Unit",
    "Quantity",
    "Unit Price",
    "Dollars",
    "Prev Qty",
    "This Period Qty",
    "Qty To Date",
    "Prev Earned",
    "This Period Earned",
    "Earned To Date",
]

lines = [
    line.strip()
    for line in table_block.splitlines()
    if line.strip()
]

EXPECTED_NUMBERS = 9
rows = []

for line in lines:

    if not line[0].isdigit():
        continue

    tokens = line.split()

    # ---- Pay Item No (numbers only)
    pay_item_tokens = []
    for tok in tokens:
        if tok.replace(".", "").isdigit():
            pay_item_tokens.append(tok)
        else:
            break
    pay_item_no = " ".join(pay_item_tokens)

    # ---- Unit
    unit_match = re.search(r"\b(ED|EA|FD|LC)\b", line)
    if not unit_match:
        continue
    unit = unit_match.group(1)

    # ---- Description (between Pay Item No and UNIT)
    try:
        unit_index = tokens.index(unit)
        description = " ".join(tokens[len(pay_item_tokens):unit_index])
    except ValueError:
        description = None

    # ---- Tail (after UNIT)
    tail = line.split(unit, 1)[1]

    # ---- POSITION-AWARE numeric extraction (KEY FIX)
    raw_tokens = re.findall(
        r"\$?\s*[\d,]+\.\d+|\$?\s*\b\d+\b|\-",
        tail
    )

    numbers = []
    for tok in raw_tokens:
        tok = tok.strip()
        if tok == "-":
            numbers.append(None)
        else:
            numbers.append(clean_number(tok))

    # ---- Pad if OCR drops columns
    if len(numbers) < EXPECTED_NUMBERS:
        numbers += [None] * (EXPECTED_NUMBERS - len(numbers))

    (
        quantity,
        unit_price,
        dollars,
        prev_qty,
        this_period_qty,
        qty_to_date,
        prev_earned,
        this_period_earned,
        earned_to_date
    ) = numbers[:9]

    rows.append({
        "Pay Item No": pay_item_no,
        "Description": description,
        "Unit": unit,
        "Quantity": quantity,
        "Unit Price": unit_price,
        "Dollars": dollars,
        "Prev Qty": prev_qty,
        "This Period Qty": this_period_qty,
        "Qty To Date": qty_to_date,
        "Prev Earned": prev_earned,
        "This Period Earned": this_period_earned,
        "Earned To Date": earned_to_date,
    })


df_table = pd.DataFrame(rows, columns=columns)
df_table

,Pay Item No,Description,Unit,Quantity,Unit Price,Dollars,Prev Qty,This Period Qty,Qty To Date,Prev Earned,This Period Earned,Earned To Date
0,30 0102 60,WORK ZONE SIGN,ED,21034,0.20,4206.80,58176.00,2380.00,60556.0000,11635.20,476.00,12111.20
1,35 0102 61,BUSINESS SIGN,EA,25,25.00,625.00,NaN,NaN,NaN,NaN,NaN,NaN
2,40 0102 74 1,"CHANNELIZING DEVICE- TYPES I, Il, Dl, V",ED,119848,0.09,10786.32,104334.00,525.00,104859.0000,9390.06,47.25,3
3,45 010274 8,CHANNELIZING DEVICE- PEDESTRIAN,LC,784,0.25,196.00,NaN,NaN,NaN,NaN,NaN,NaN
4,50 0102 76,ARROW BOARD / ADVANCE WARNING Af,ED,437,5.00,2185.00,811.00,NaN,811.0000,4055.00,NaN,4055.00
5,55 0102 99,PORTABLE CHANGEABLE MESSAGE SIG,ED,1456,9.00,13104.00,1655.00,70.00,1725.0000,14895.00,630.00,15525.00
6,70 102115,TYPE Il BARRICADE,ED,700,0.25,175.00,NaN,NaN,NaN,NaN,NaN,NaN
7,75 0102150 1,"PORTABLE REGULATORY, SIGN",ED,1400,400,5600.00,1240.00,NaN,1240.0000,4960.00,NaN,4960.00
8,80 0102150 2,RADAR SPEED DISPLAY UNIT,ED,1400,4.00,5600.00,1260.00,NaN,1260.0000,5040.00,5040.00,NaN


In [18]:
import numpy as np

numeric_cols = [
    "Quantity",
    "Unit Price",
    "Dollars",
    "Prev Qty",
    "This Period Qty",
    "Qty To Date",
    "Prev Earned",
    "This Period Earned",
    "Earned To Date",
]

for c in numeric_cols:
    df_table[c] = pd.to_numeric(df_table[c], errors="coerce")

In [19]:
def reconcile_row(row, tol=0.01):
    mismatches = []

    qty = row["Quantity"]
    price = row["Unit Price"]
    dollars = row["Dollars"]

    prev_qty = row["Prev Qty"]
    period_qty = row["This Period Qty"]
    qty_to_date = row["Qty To Date"]

    prev_earned = row["Prev Earned"]
    period_earned = row["This Period Earned"]
    earned_to_date = row["Earned To Date"]

    # ---- Dollars vs Qty × Unit Price
    if pd.notna(qty) and pd.notna(price):
        calc_dollars = qty * price
        if pd.notna(dollars) and abs(dollars - calc_dollars) > tol:
            mismatches.append("Dollars mismatch (Qty × Unit Price)")
        elif pd.isna(dollars):
            row["Dollars"] = calc_dollars

    # ---- Unit Price vs Dollars ÷ Quantity
    if pd.notna(dollars) and pd.notna(qty) and qty != 0:
        inferred_price = dollars / qty
        if pd.notna(price) and abs(price - inferred_price) > tol:
            mismatches.append("Unit Price mismatch (Dollars ÷ Quantity)")
        elif pd.isna(price):
            row["Unit Price"] = inferred_price

    # ---- Qty To Date vs Prev + Period
    if pd.notna(prev_qty) and pd.notna(period_qty):
        calc_qty_to_date = prev_qty + period_qty
        if pd.notna(qty_to_date) and abs(qty_to_date - calc_qty_to_date) > tol:
            mismatches.append("Qty To Date mismatch (Prev + This Period)")
        elif pd.isna(qty_to_date):
            row["Qty To Date"] = calc_qty_to_date

    # ---- Prev Earned vs Prev Qty × Unit Price
    if pd.notna(prev_qty) and pd.notna(price):
        calc_prev_earned = prev_qty * price
        if pd.notna(prev_earned) and abs(prev_earned - calc_prev_earned) > tol:
            mismatches.append("Prev Earned mismatch (Prev Qty × Unit Price)")
        elif pd.isna(prev_earned):
            row["Prev Earned"] = calc_prev_earned

    # ---- Earned To Date vs Prev + Period
    if pd.notna(prev_earned) and pd.notna(period_earned):
        calc_earned = prev_earned + period_earned
        if pd.notna(earned_to_date) and abs(earned_to_date - calc_earned) > tol:
            mismatches.append("Earned To Date mismatch (Prev + This Period)")
        elif pd.isna(earned_to_date):
            row["Earned To Date"] = calc_earned

    row["Mismatch_Found"] = len(mismatches) > 0
    row["Mismatch_Details"] = "; ".join(mismatches)

    return row

In [21]:
df_table_reconciled = df_table.apply(
    reconcile_row,
    axis=1
)
df_table_reconciled

,Pay Item No,Description,Unit,Quantity,Unit Price,Dollars,Prev Qty,This Period Qty,Qty To Date,Prev Earned,This Period Earned,Earned To Date,Mismatch_Found,Mismatch_Details
0,30 0102 60,WORK ZONE SIGN,ED,21034,0.20,4206.80,58176.0,2380.0,60556.0,11635.20,476.00,12111.2,False,
1,35 0102 61,BUSINESS SIGN,EA,25,25.00,625.00,NaN,NaN,NaN,NaN,NaN,NaN,False,
2,40 0102 74 1,"CHANNELIZING DEVICE- TYPES I, Il, Dl, V",ED,119848,0.09,10786.32,104334.0,525.0,104859.0,9390.06,47.25,3.0,True,Earned To Date mismatch (Prev + This Period)
3,45 010274 8,CHANNELIZING DEVICE- PEDESTRIAN,LC,784,0.25,196.00,NaN,NaN,NaN,NaN,NaN,NaN,False,
4,50 0102 76,ARROW BOARD / ADVANCE WARNING Af,ED,437,5.00,2185.00,811.0,NaN,811.0,4055.00,NaN,4055.0,False,
5,55 0102 99,PORTABLE CHANGEABLE MESSAGE SIG,ED,1456,9.00,13104.00,1655.0,70.0,1725.0,14895.00,630.00,15525.0,False,
6,70 102115,TYPE Il BARRICADE,ED,700,0.25,175.00,NaN,NaN,NaN,NaN,NaN,NaN,False,
7,75 0102150 1,"PORTABLE REGULATORY, SIGN",ED,1400,400.00,5600.00,1240.0,NaN,1240.0,4960.00,NaN,4960.0,True,Dollars mismatch (Qty × Unit Price); Unit Pric...
8,80 0102150 2,RADAR SPEED DISPLAY UNIT,ED,1400,4.00,5600.00,1260.0,NaN,1260.0,5040.00,5040.00,10080.0,False,


In [15]:
df_table_enriched = df_table.assign(
    Job_Number=df_metadata.loc[0, "Job Number"],
    Job_Name=df_metadata.loc[0, "Job Name"],
    Job_Address=df_metadata.loc[0, "Job Address"],
    Invoice_Amount=df_metadata.loc[0, "Amount"]
)

#### Anderson Columbia

In [16]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import re

# 🔧 CONFIG
PDF_PATH = r"C:\Users\IlaBarshilia\OneDrive - ldiltd.com\M1 Global - M1 Global\Projects\Job Reconciliation Analysis\PDF Input from Top Customers\Anderson Columbia.pdf"
OUTPUT_TEXT_FILE = r"C:\Users\IlaBarshilia\OneDrive - ldiltd.com\M1 Global - M1 Global\Projects\Job Reconciliation Analysis\PDF Input from Top Customers\Anderson Columbia_raw_text.txt"

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Users\IlaBarshilia\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"
)

POPPLER_PATH = r"C:\Users\IlaBarshilia\Downloads\Software\Poppler\poppler-25.12.0\Library\bin"

# Convert PDF → Images
pages = convert_from_path(PDF_PATH, dpi=300, poppler_path=POPPLER_PATH)

def correct_rotation(image):
    """Detect and fix rotation using Tesseract OSD"""
    try:
        osd = pytesseract.image_to_osd(image)
        rotate_angle = int(re.search(r"Rotate: (\d+)", osd).group(1))
        if rotate_angle != 0:
            image = image.rotate(-rotate_angle, expand=True)
    except Exception:
        pass
    return image

all_text = []

for idx, page in enumerate(pages, start=1):
    print(f"OCR page {idx}...")

    page = correct_rotation(page)

    text = pytesseract.image_to_string(
        page,
        config="--psm 6"  # good for tables/invoices
    )
    all_text.append(text)

# Save raw OCR output
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_text))

print("✅ Raw OCR text saved:", OUTPUT_TEXT_FILE)

OCR page 1...
OCR page 2...
OCR page 3...
OCR page 4...
OCR page 5...
OCR page 6...
OCR page 7...
OCR page 8...
OCR page 9...
OCR page 10...
OCR page 11...
OCR page 12...
OCR page 13...
✅ Raw OCR text saved: C:\Users\IlaBarshilia\OneDrive - ldiltd.com\M1 Global - M1 Global\Projects\Job Reconciliation Analysis\PDF Input from Top Customers\Anderson Columbia_raw_text.txt
